# 2024年版ものづくり白書（全340ページ・31MB）を対象としたRAG構築

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

本ノートブックでは、**「2024年版ものづくり白書（令和5年度ものづくり基盤技術の振興施策）」**を題材に、検索精度を高めるための3段階アプローチを実装・検証します。

### 🎯 本ノートブックの学習目標
1. **大容量白書のチャンキング設計**: 1節単位（数千〜数万文字）の肥大化を防ぎ、**章・節メタデータを付与しながら600〜800文字に分割**する技法
2. **段階的な検索エンジンの実装**:
   - **① キーワード検索 (BM25)**: 「価格転嫁」「GX」などの固有名詞・専門用語をピンポイント検索
   - **② ベクトル検索 (ChromaDB + gemini-embedding-001)**: 類義語やニュアンスでの意味的検索（無料枠のレートリミットを回避するバッチ分割登録）
   - **③ ハイブリッド検索 (RRF: Reciprocal Rank Fusion)**: ①と②の順位を融合して最高精度を実現
3. **出典（章・節・ページ番号）付き回答生成**: Gemini 2.5 Flashによる根拠付きQA
4. **Gradio Web UI**: Colabセル内での操作画面と共有URLの発行

## 1. 必要なライブラリのインストール
キーワード検索用の `rank-bm25` を含む必要ライブラリをインストールします。キーワード検索用の `rank-bm25` やベクトルDB用の `langchain-chroma` を含む必要ライブラリをインストールします。

In [1]:
!pip install -q --no-warn-conflicts \
    langchain \
    langchain-community \
    langchain-google-genai \
    langchain-chroma \
    chromadb \
    pypdf \
    gradio \
    rank-bm25

print("✅ ライブラリのインストールが完了しました！")

✅ ライブラリのインストールが完了しました！


## 2. Gemini API キーの設定
Colab画面左側の 🔑「シークレット（Secrets）」に `GEMINI_API_KEY` を登録し、「ノートブックへのアクセス」をONにしてください。

In [2]:
import os
from google.colab import userdata

try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("✅ GEMINI_API_KEY をColabシークレットから正常に設定しました。")
except Exception:
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API Key を入力してください: ")

✅ GEMINI_API_KEY をColabシークレットから正常に設定しました。


## 3. 白書PDF（31MB）の配置
経産省の公式サイトからダウンロードするか、GitHubにコミットしたPDFを配置します。

In [4]:
import os
os.makedirs("data", exist_ok=True)
pdf_target = "data/document.pdf"

# まだファイルがない場合は経産省サイトから直接ダウンロード
if not os.path.exists(pdf_target):
    print(f"[*] {pdf_target} をダウンロード中...")
    !wget -q https://www.meti.go.jp/report/whitepaper/mono/2024/honbun/pdf/all.pdf -O data/document.pdf

if os.path.exists(pdf_target):
    size_mb = os.path.getsize(pdf_target) / (1024*1024)
    print(f"✅ 白書PDFを確認しました: {pdf_target} ({size_mb:.2f} MB)")
else:
    print("※ 左メニューの📁ファイルから31MBの白書PDFをドラッグ＆ドロップして 'data/document.pdf' に配置してください。")

✅ 白書PDFを確認しました: data/document.pdf (30.95 MB)


## 4. 階層メタデータ（章・節・ページ）付きチャンク分割 & pandas集計
白書の「1節まるごとだと長すぎる」問題を解決するため、**章・節見出しをメタデータとして保持しながら指定した文字数単位に分割**します。

> 💡 **右側のフォーム（スライダー）で  や  を直感的に変更できます。**
> チャンクサイズを変更した場合は、このセルを実行後、後続の検索エンジン構築セル（5-1, 5-2, 5-3）を再実行してください。

In [5]:
#@title ⚙️ チャンク分割パラメータの設定 { run: "auto" }
CHUNK_SIZE = 700      #@param {type:"slider", min:300, max:1500, step:50}
CHUNK_OVERLAP = 150   #@param {type:"slider", min:50, max:400, step:10}

import re
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def extract_headings(text, current):
    meta = current.copy()
    p_match = re.search(r'(第[一二三四五六七八九十\d]+部\s+[^\n]{2,30})', text)
    if p_match: meta['part'] = p_match.group(1).strip()
    c_match = re.search(r'(第[一二三四五六七八九十\d]+章\s+[^\n]{2,35})', text)
    if c_match: meta['chapter'] = c_match.group(1).strip()
    s_match = re.search(r'(第[一二三四五六七八九十\d]+節\s+[^\n]{2,35})', text)
    if s_match: meta['section'] = s_match.group(1).strip()
    return meta

if os.path.exists(pdf_target):
    print(f"[*] チャンク分割を実行中... (CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP})")
    loader = PyPDFLoader(pdf_target)
    pages = loader.load()
    print(f"📖 読込ページ数: {len(pages)} ページ")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", "。", "、", " ", ""]
    )

    chunks = []
    records = []
    curr_meta = {'part': '第1部', 'chapter': '第1章', 'section': '第1節'}
    chunk_id = 0

    for p_doc in pages:
        p_num = p_doc.metadata.get('page', 0) + 1
        curr_meta = extract_headings(p_doc.page_content, curr_meta)
        p_chunks = splitter.split_text(p_doc.page_content)

        for txt in p_chunks:
            if len(txt.strip()) < 30: continue
            chunk_id += 1
            meta = {
                'chunk_id': chunk_id,
                'page': p_num,
                'part': curr_meta['part'],
                'chapter': curr_meta['chapter'],
                'section': curr_meta['section']
            }
            chunks.append(Document(page_content=txt.strip(), metadata=meta))
            records.append({
                'chunk_id': chunk_id,
                'page': p_num,
                'section': curr_meta['section'],
                'char_count': len(txt.strip()),
                'preview': txt.strip()[:60].replace('\n', ' ') + '...'
            })

    df_chunks = pd.DataFrame(records)
    print(f"✂️ 分割チャンク数: {len(chunks)} 件 (CHUNK_SIZE={CHUNK_SIZE})")
    print("\n📊 --- チャンクの文字数要約統計 (pandas describe) ---")
    display(df_chunks['char_count'].describe())
    print("\n📋 --- 検出された節ごとのチャンク数 (Top 5) ---")
    display(df_chunks['section'].value_counts().head(5))
    print("\n💡 【ご案内】チャンクサイズを変更した場合は、次のセル（5-1, 5-2, 5-3）を再実行して検索インデックスを更新してください。")
else:
    print("※ PDF配置後にこのセルを実行してください。")

/tmp/ipykernel_5965/1094780176.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


📖 読込ページ数: 340 ページ
✂️ 分割チャンク数: 714 件

📊 --- チャンクの文字数要約統計 (pandas describe) ---


,char_count
count,714.000000
mean,561.696078
std,169.717714
min,42.000000
25%,455.250000
50%,661.500000
75%,687.000000
max,700.000000



📋 --- 検出された節ごとのチャンク数 (Top 5) ---


,count
section,
第1節 人材確保と雇用の安定,629
第3節 ものづくりに関する能力の適正な評価、労働条件の確保・改善,57
第2節\n・内閣府［2023］ 『令和5年度 年次経済財政報告（経済財政政策担,6
第1節,5
第2節\n・ 経済産業省［2023］ 『産業構造審議会,3


## 5-1. 【ステップ1】キーワード検索 (BM25) の実装
日本語トークナイズ（単語＋2-gram）とBM25アルゴリズムを用いて、専門用語の完全一致検索を実装します。

In [6]:
from rank_bm25 import BM25Okapi

def tokenize_jp(text):
    clean = text.lower()
    words = re.findall(r'[a-zA-Z0-9]+|[一-龥]{2,}|[ァ-ンヴー]{2,}', clean)
    bigrams = [clean[i:i+2] for i in range(len(clean)-1) if not re.match(r'\s', clean[i:i+2])]
    return list(set(words + bigrams)) or [clean]

if 'chunks' in locals() and len(chunks) > 0:
    print("[*] BM25インデックスを作成中...")
    corpus_tokens = [tokenize_jp(c.page_content) for c in chunks]
    bm25 = BM25Okapi(corpus_tokens)
    print("✅ BM25キーワード検索エンジンの準備が完了しました。")

    # 単体検索テスト
    test_kw = "価格転嫁"
    scores = bm25.get_scores(tokenize_jp(test_kw))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:3]
    print(f"\n🔎 『{test_kw}』のBM25検索結果 (Top 3):")
    for r, idx in enumerate(top_idx, 1):
        m = chunks[idx].metadata
        print(f"  {r}. {m['section']} (p.{m['page']}) スコア: {scores[idx]:.2f}")
        print(f"     {chunks[idx].page_content[:80]}...")

[*] BM25インデックスを作成中...
✅ BM25キーワード検索エンジンの準備が完了しました。

🔎 『価格転嫁』のBM25検索結果 (Top 3):
  1. 第1節　人材確保と雇用の安定 (p.286) スコア: 17.86
     ③ コストが上昇せず、価格交渉は不要と判断し、発注企業からの申し⼊れを辞退した。
④ コストが上昇せず、発注企業から申し⼊れはなかったが、価格交渉は不要と判断し...
  2. 第1節　人材確保と雇用の安定 (p.32) スコア: 17.05
     理解が得にくい／取引先との交渉力が弱い」を挙げる割合が最も大きい（図 120-11） 。
図120-11　価格転嫁が進んでいない要因
71.3
12.9
11....
  3. 第1節　人材確保と雇用の安定 (p.31) スコア: 16.90
     次に、製造事業者が直 近3年 間で実施した企業行動について「プラス・マイナス」の影響
1
の有無に関する調査によれば、中小企業では、 「賃上げ（従業員への還元）...


## 5-2. 【ステップ2】ベクトル検索 (ChromaDB + ローカル多言語モデル) の実装
Google APIの制限（1日1,000回上限など）を完全に回避するため、Colab上で動作するオープンソースの日本語多言語対応モデル `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384次元) を使用します。
※ API通信を行わずローカル（CPU/GPU）で処理するため、レートリミットや利用制限エラーなしで全チャンクを数秒〜数十秒で安全にインデックス化できます。

In [7]:
import os
import shutil
import torch

os.environ["ANONYMIZED_TELEMETRY"] = "False"  # ChromaDBのテレメトリ警告を抑止
from langchain_chroma import Chroma

# 1. 完全ローカルの多言語埋め込みモデル（API制限・クォータ制限ゼロ）
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] ローカル埋め込みモデルをロード中 (Device: {device})...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)
print("✅ ローカル埋め込みモデルの準備完了 (384次元 / API制限なし)")

if 'chunks' in locals() and len(chunks) > 0:
    total_chunks = len(chunks)
    print(f"[*] ChromaDB に全 {total_chunks} 件を高速インデックス中 (完全ローカル処理)...\n")

    # チャンクサイズ変更時に古いインデックスと混ざらないよう再作成
    persist_dir = "./chroma_db_local"
    if os.path.exists(persist_dir):
        shutil.rmtree(persist_dir, ignore_errors=True)

    vector_store = Chroma(
        collection_name="whitepaper_vectors_local",
        embedding_function=embeddings,
        persist_directory=persist_dir
    )

    # API制限がないため、100件ずつの高速一括登録が可能
    BATCH_SIZE = 100
    for i in range(0, total_chunks, BATCH_SIZE):
        batch = chunks[i : i + BATCH_SIZE]
        vector_store.add_documents(batch)
        done_count = min(i + len(batch), total_chunks)
        print(f"  [+] 進捗: {done_count}/{total_chunks} 件インデックス完了")

    print("\n✅ ベクトル検索エンジンの準備が完了しました！")

    # 単体検索テスト（言い換え表現で検索）
    test_query = "原材料費の高騰を取引価格に反映できているか？"
    vec_results = vector_store.similarity_search_with_relevance_scores(test_query, k=3)
    print(f"\n🔎 『{test_query}』のベクトル検索結果 (Top 3):")
    for r, (doc, sc) in enumerate(vec_results, 1):
        m = doc.metadata
        print(f"  {r}. {m['section']} (p.{m['page']}) 類似度: {sc:.3f}")
        print(f"     {doc.page_content[:80]}...")


[*] ローカル埋め込みモデルをロード中 (Device: cpu)...


/tmp/ipykernel_5965/1855898036.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ ローカル埋め込みモデルの準備完了 (384次元 / API制限なし)
[*] ChromaDB に全 714 件を高速インデックス中 (完全ローカル処理)...

  [+] 進捗: 100/714 件インデックス完了
  [+] 進捗: 200/714 件インデックス完了
  [+] 進捗: 300/714 件インデックス完了
  [+] 進捗: 400/714 件インデックス完了
  [+] 進捗: 500/714 件インデックス完了
  [+] 進捗: 600/714 件インデックス完了
  [+] 進捗: 700/714 件インデックス完了
  [+] 進捗: 714/714 件インデックス完了

✅ ベクトル検索エンジンの準備が完了しました！

🔎 『原材料費の高騰を取引価格に反映できているか？』のベクトル検索結果 (Top 3):
  1. 第1節　人材確保と雇用の安定 (p.29) 類似度: 0.552
     原材料価格・エネルギー価格の高騰などの事業環境変化の中で、製造事業者が直近 3 年間
で実施した企業行動に関する調査によれば、半数以上の企業が「価格転嫁（販売先...
  2. 第1節　人材確保と雇用の安定 (p.32) 類似度: 0.552
     2023 年度において、製造・販売する製品や部素材の原材料調達における高騰分の価格転
嫁の割合をみると、 「80％以上～90％未満」を挙げた事業者が最も多く、続...
  3. 第1節　人材確保と雇用の安定 (p.27) 類似度: 0.511
     事業に影響を及ぼす社会情勢の変化に関する調査について、2022 年度に行われた同様の
調査結果と比較すると、2022 年度に引き続き 2023 年度も「原材料価...


## 5-3. 【ステップ3】ハイブリッド検索 (Reciprocal Rank Fusion: RRF) の実装
BM25の順位とベクトルの順位をRRF式で融合し、専門用語と意味理解の双方で優れた総合ランキングを算出します。

In [8]:
def hybrid_search(query, top_k=4, rrf_k=60, w_kw=0.5, w_vec=0.5):
    # 1. BM25 検索
    kw_scores = bm25.get_scores(tokenize_jp(query))
    kw_top = sorted(range(len(kw_scores)), key=lambda i: kw_scores[i], reverse=True)[:top_k*2]

    # 2. ベクトル検索
    vec_res = vector_store.similarity_search_with_relevance_scores(query, k=top_k*2)

    fused = {}
    # BM25 RRF加算
    for rank, idx in enumerate(kw_top, 1):
        if kw_scores[idx] <= 0: continue
        c_id = chunks[idx].metadata['chunk_id']
        fused[c_id] = {
            'doc': chunks[idx],
            'score': w_kw * (1.0 / (rrf_k + rank)),
            'kw_rank': rank,
            'vec_rank': None
        }

    # ベクトル RRF加算
    for rank, (doc, sc) in enumerate(vec_res, 1):
        c_id = doc.metadata['chunk_id']
        score_val = w_vec * (1.0 / (rrf_k + rank))
        if c_id in fused:
            fused[c_id]['score'] += score_val
            fused[c_id]['vec_rank'] = rank
        else:
            fused[c_id] = {'doc': doc, 'score': score_val, 'kw_rank': None, 'vec_rank': rank}

    ranked = sorted(fused.values(), key=lambda x: x['score'], reverse=True)[:top_k]
    return ranked

if 'vector_store' in locals():
    compare_q = "ものづくり企業のGX・脱炭素化の取り組み"
    print(f"\n🏆 『{compare_q}』のハイブリッド検索 (Top 3):")
    for i, item in enumerate(hybrid_search(compare_q, top_k=3), 1):
        m = item['doc'].metadata
        print(f"  {i}. {m['section']} (p.{m['page']}) RRFスコア: {item['score']*100:.2f} (BM25順位:{item['kw_rank']}, ベクトル順位:{item['vec_rank']})")
        print(f"     {item['doc'].page_content[:80]}...")


🏆 『ものづくり企業のGX・脱炭素化の取り組み』のハイブリッド検索 (Top 3):
  1. 第1節　人材確保と雇用の安定 (p.264) RRFスコア: 0.82 (BM25順位:1, ベクトル順位:None)
     課題である。このため、CO2 を原料として利用し CO2 排出削減・固定量を最大化するコン
クリートの開発、 より低コストな製造 ・ 現場施工技術の開発、 コン...
  2. 第1節　人材確保と雇用の安定 (p.181) RRFスコア: 0.82 (BM25順位:None, ベクトル順位:1)
     献を目指し、従来の延長線上にない非連続なイノベーションをもたらす革新的技術の創出に
向けた研究開発を競争的環境下で推進しており、2023 年度からは新たに「革新...
  3. 第1節　人材確保と雇用の安定 (p.265) RRFスコア: 0.81 (BM25順位:2, ベクトル順位:None)
     等の電気配線を光配線化する革新的な光電融合技術により、データ集約拠点であるデータセ
ンターの 40％以上の大幅な省エネ化を目指した開発を推進している。2023 ...


## 6. Gemini 2.5 Flash による出典（章・節・ページ）付き回答生成
ハイブリッド検索結果をプロンプトに挿入し、根拠のある回答を生成します。

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 最新の推奨モデルである gemini-3.8-flash を指定
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.2)

# 出典明記プロンプトテンプレート（章・節ごとのグルーピング仕様）
prompt_tmpl = """あなたは「ものづくり白書」の専門アシスタントです。
以下の【資料】のみを根拠にして質問に回答してください。
資料に記載のない内容は推測せず「資料に記載がありません」と答え、回答文中に参照箇所の [第X章 第X節 p.Y] を明記してください。

また、回答の末尾には必ず「---」で区切り、実際に回答作成で参照した資料を以下の【出力フォーマット】に従って章・節ごとにページ番号をグルーピングして一覧化してください。
回答の「---」より上の内容には、章・節・ページ番号は記載しないでください。(回答の見やすさ重視のため)

【出力フォーマットのルール】:
1. 同じ「章・節」の資料が複数ある場合は、1行にまとめてページ番号を「p.Y1, Y2」の形式で昇順（小さい順）に並べてください。
2. 回答作成に利用しなかった無関係な資料はリストに含めないでください。

---
**📚 参照した資料:**
- [章名] [節名] [p.88, 226, 269]
- [章名] [節名] [p.328]

【資料】:
{context}

【質問】:
{question}

【回答】:"""

def ask_whitepaper(question, mode='hybrid'):
    hits = hybrid_search(question, top_k=4)
    context_parts = []
    for i, h in enumerate(hits, 1):
        m = h['doc'].metadata
        loc = f"{m.get('chapter', '')} {m.get('section', '')} [p.{m.get('page')}]"
        context_parts.append(f"--- [資料{i}: {loc}] ---\n{h['doc'].page_content}")
    context_text = "\n\n".join(context_parts)
    chain = PromptTemplate.from_template(prompt_tmpl) | llm | StrOutputParser()
    return chain.invoke({"context": context_text, "question": question}), hits

if 'vector_store' in locals():
    sample_q = "中小製造業における価格転嫁の現状と直面している課題について教えてください。"
    print(f"❓ 質問: {sample_q}\n")
    answer, used_hits = ask_whitepaper(sample_q)
    print(f"💡 回答:\n{answer}")

❓ 質問: 中小製造業における価格転嫁の現状と直面している課題について教えてください。

💡 回答:
資料に基づき、中小企業における価格転嫁の現状と直面している課題について説明します。

### 価格転嫁・価格交渉の現状
* **交渉を切り出せない・応じてもらえない実態**  
  コストが上昇している場合でも、「発注減少や取引停止を恐れ、発注企業からの申し入れを辞退した」「発注減少や取引停止を恐れ、受注企業から交渉を申し出なかった」「受注企業から交渉を申し出たが、応じてもらえなかった」といった状況が存在しています [第2章 第1節 p.286]。
* **芳しくない親事業者への対応**  
  フォローアップ調査の結果を踏まえ、交渉と転嫁の状況が芳しくない親事業者（累計約40社）に対し、「下請中小企業振興法」に基づき事業所管大臣名での「指導・助言」が実施されています [第2章 第1節 p.287]。

### 直面している課題（価格転嫁が進んでいない要因）
中小企業において価格転嫁が進んでいない要因として、以下の点が挙げられています [第2章 第1節 p.32]：
* **取引先からの理解が得にくい／取引先との交渉力が弱い（68.2%）**：最も割合が大きくなっています。
* **取引先と長期契約を結んでいるため価格改定が困難（18.2%）**
* **市場での競争力が弱い（13.6%）**
* **消費者からの理解が得にくい（13.6%）**
* **そもそも価格転嫁の必要がない（4.5%）**
* **その他（22.7%）**

※なお、提供された資料以外の詳細な現状や要因については資料に記載がありません。


## 7. Gradio Web チャットボットUI（検索方式切替付き）の起動
3つの検索エンジンを自由に切り替えながら対話テストできるUIを起動します。

In [25]:
import gradio as gr
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Gemini 3.8 Flash モデルの準備
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.2)

# 2. 出典明記プロンプトテンプレート
prompt_tmpl = """あなたは「ものづくり白書」の専門アシスタントです。
以下の【資料】のみを根拠にして質問に回答してください。
資料に記載のない内容は推測せず「資料に記載がありません」と答え、回答文中に参照箇所の [第X章 第X節 p.Y] を明記してください。

また、回答の末尾には必ず「---」で区切り、実際に回答作成で参照した資料を以下の【出力フォーマット】に従って章・節ごとにページ番号をグルーピングして一覧化してください。
回答の「---」より上の内容には、章・節・ページ番号は記載しないでください。(回答の見やすさ重視のため)

【出力フォーマットのルール】:
1. 同じ「章・節」の資料が複数ある場合は、1行にまとめてページ番号を「p.Y1, Y2」の形式で昇順（小さい順）に並べてください。
2. 回答作成に利用しなかった無関係な資料はリストに含めないでください。

---
**📚 参照した資料:**
- [章名] [節名] [p.88, 226, 269]
- [章名] [節名] [p.328]

【資料】:
{context}

【質問】:
{question}

【回答】:"""

# 3. チャットイベントハンドラ (v1で動作確認済みの安定仕様)
def chat_handler(message, history, search_mode):
    # 履歴（history）の初期化
    dict_history = list(history) if history else []

    if 'vector_store' not in globals() or 'bm25' not in globals():
        dict_history.append({"role": "user", "content": message})
        dict_history.append({"role": "assistant", "content": "⚠️ 先に検索エンジン構築セルを実行してください。"})
        return dict_history

    if not message.strip():
        return dict_history

    # 検索の実行（上位4件取得）
    if search_mode == "キーワード検索 (BM25)":
        kw_scores = bm25.get_scores(tokenize_jp(message))
        top_idx = sorted(range(len(kw_scores)), key=lambda i: kw_scores[i], reverse=True)[:4]
        hits = [{'doc': chunks[i], 'kw_rank': r, 'vec_rank': None} for r, i in enumerate(top_idx, 1)]
    elif search_mode == "ベクトル検索 (ChromaDB)":
        vec_res = vector_store.similarity_search_with_relevance_scores(message, k=4)
        hits = [{'doc': doc, 'kw_rank': None, 'vec_rank': r} for r, (doc, _) in enumerate(vec_res, 1)]
    else:
        hits = hybrid_search(message, top_k=4)

    # 資料テキストの結合
    context_parts = []
    for i, h in enumerate(hits, 1):
        m = h['doc'].metadata
        loc = f"{m.get('chapter', '')} {m.get('section', '')} [p.{m.get('page')}]"
        context_parts.append(f"--- [資料{i}: {loc}] ---\n{h['doc'].page_content}")

    # =================================================================
    # 【変更点】Python側での sources_line の強制結合をやめ、
    # Geminiがプロンプトの指示に従って出力した回答をそのまま採用する
    # =================================================================
    chain = PromptTemplate.from_template(prompt_tmpl) | llm | StrOutputParser()
    reply = chain.invoke({"context": "\n\n".join(context_parts), "question": message})

    # メッセージ履歴に追加して返却
    dict_history.append({"role": "user", "content": message})
    dict_history.append({"role": "assistant", "content": reply})
    return dict_history

# 4. Gradio UIの構築と起動 (Colab環境で型エラーの出ない安全な構成)
with gr.Blocks(title="2024年版ものづくり白書 RAG") as demo:
    gr.Markdown("# 📘 2024年版ものづくり白書 RAG チャットボット")
    gr.Markdown("**3段階検索切替: ① キーワード検索(BM25) / ② ベクトル検索(ChromaDB) / ③ ハイブリッド検索(RRF)**")

    mode_select = gr.Radio(
        choices=["ハイブリッド検索 (RRF統合) [推奨]", "キーワード検索 (BM25)", "ベクトル検索 (ChromaDB)"],
        value="ハイブリッド検索 (RRF統合) [推奨]",
        label="検索方式の選択"
    )
    chatbot_ui = gr.Chatbot(height=480, label="対話履歴 (出典付き回答)")
    msg = gr.Textbox(placeholder="白書について質問を入力 (例: 価格転嫁の進捗状況と課題について教えてください)...", label="質問入力")

    with gr.Row():
        send_btn = gr.Button("送信", variant="primary", scale=4)
        clear_btn = gr.Button("対話クリア", scale=1)

    # 送信時のイベント設定 (v1と同等)
    send_btn.click(chat_handler, [msg, chatbot_ui, mode_select], chatbot_ui)
    send_btn.click(lambda: "", None, msg)

    msg.submit(chat_handler, [msg, chatbot_ui, mode_select], chatbot_ui)
    msg.submit(lambda: "", None, msg)

    clear_btn.click(lambda: [], None, chatbot_ui)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a737f961360dfc902f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a737f961360dfc902f.gradio.live
